# Output Parsers and Structured Output

Chat models return **`AIMessage`** objects — rich structures with content, metadata, and optional tool calls. Your API, database, and UI usually need **plain strings**, **JSON**, or **typed Pydantic objects**. **Output parsers** are LangChain Runnables that sit at the end of a chain and normalize model output into the shape downstream code expects.

This notebook covers **`StrOutputParser`**, **`JsonOutputParser`**, **`PydanticOutputParser`**, list and XML parsers, **`.with_structured_output()`** on chat models, combining parsers with prompt format instructions (**04. Prompt Templates**), handling parse failures, streaming through parsers, and choosing a structured-output strategy for production.

**03. Chat Models and Messages** previewed `AIMessage` fields and `with_structured_output`. Here we treat parsing as a first-class chain step: `prompt | llm | parser`.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core
import numpy as np

np.set_printoptions(precision=4, suppress=True)

print("langchain-core:", langchain_core.__version__)

---

## 1. The Parsing Problem

Without a parser, the last step of `prompt | llm` returns an **`AIMessage`**:

```python
ai_message.content  # str — but wrapped in AIMessage
ai_message.response_metadata  # token usage, finish_reason
ai_message.tool_calls  # possibly non-empty
```

| Consumer | Wants | Without parser |
|----------|-------|----------------|
| FastAPI response | `str` or `dict` | Manual `.content` extraction |
| SQL insert | Validated Pydantic model | Regex on raw text |
| Downstream chain step | Typed object | Fragile string parsing |
| JSON API client | JSON-serializable dict | May get markdown fences around JSON |

**Output parsers** encapsulate extraction and validation at the **chain boundary** — one place to fix when models change formatting habits.

---

## 2. Output Parsers as Runnables

All parsers implement **`BaseOutputParser`** and the **Runnable** protocol (**02. Runnable Protocol and LCEL**).

```
AIMessage (or str) ──► OutputParser.invoke() ──► str | dict | BaseModel | list
```

| Method | Role |
|--------|------|
| **`invoke(input)`** | Parse one model output |
| **`parse(text)`** | Direct parse on string (convenience) |
| **`get_format_instructions()`** | Text to embed in prompts (JSON/XML parsers) |
| **`OutputParserException`** | Raised when parsing fails |

In LCEL: `chain = prompt | llm | parser` — the parser receives whatever the LLM step returns.

---

## 3. StrOutputParser — Plain Text

**`StrOutputParser`** is the default terminator for chat chains. It extracts **`message.content`** as a Python **`str`**.

Accepts:

- `AIMessage`
- `str` (pass-through)
- `BaseMessage` subclasses with `.content`

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

parser = StrOutputParser()

# Direct parse
ai = AIMessage(content="JWT uses Bearer tokens.")
print("from AIMessage:", parser.invoke(ai))
print("type:", type(parser.invoke(ai)).__name__)

# In chain
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
prompt = ChatPromptTemplate.from_template("Answer in one sentence: {q}")
chain = prompt | llm | StrOutputParser()

print("chain output:", chain.invoke({"q": "What is HTTPS?"}))

### 3.1 When to Use StrOutputParser

| Use | Skip |
|-----|------|
| User-facing prose answers | You need JSON or Pydantic |
| Summarization, Q&A | Model returns tool_calls only |
| RAG final answers | Structured extraction pipelines |

---

## 4. JsonOutputParser — JSON Dicts

**`JsonOutputParser`** parses model text into a Python **`dict`**. It can strip markdown code fences (\`\`\`json ... \`\`\`) and optionally validate against a **Pydantic schema**.

Best paired with **`format_instructions`** in the system prompt (**04. Prompt Templates** `.partial()`).

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


class NoteMeta(BaseModel):
    title: str = Field(description="Short title for the note")
    tags: list[str] = Field(description="Up to 3 lowercase tags")
    priority: int = Field(description="1-5 priority score")


json_parser = JsonOutputParser(pydantic_object=NoteMeta)

print("format_instructions (first 300 chars):")
print(json_parser.get_format_instructions()[:300], "...\n")

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract metadata from the user note.\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=json_parser.get_format_instructions())

extract_chain = extract_prompt | llm | json_parser

result = extract_chain.invoke({
    "text": "Deploy Notes API Friday. Blockers: security review. Tag backend."
})
print(json.dumps(result, indent=2))

### 4.1 Parsing Raw JSON Strings (Offline Test)

Test parser logic without calling the model:

In [ ]:
raw_model_output = """Here is the result:
```json
{"title": "Deploy API", "tags": ["backend", "devops"], "priority": 4}
```
"""

parsed = json_parser.parse(raw_model_output)
print(parsed)
print("validated as NoteMeta:", NoteMeta(**parsed))

---

## 5. PydanticOutputParser — Typed Models

**`PydanticOutputParser`** returns a **Pydantic `BaseModel` instance** directly instead of a raw dict. It generates format instructions from the schema and validates fields on parse.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object=NoteMeta)

pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract note metadata.\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=pydantic_parser.get_format_instructions())

pydantic_chain = pydantic_prompt | llm | pydantic_parser

note = pydantic_chain.invoke({
    "text": "Fix JWT expiry bug. High priority. Tags: auth, security."
})

print(type(note))
print(note)
print("title:", note.title, "| priority:", note.priority)

---

## 6. with_structured_output — Provider-Native Parsing

Modern chat models support **structured output** via API-level JSON schema or tool-style constraints. **`.with_structured_output(schema)`** wraps the model so **`invoke` returns the schema type directly** — no separate parser step in the chain.

```
prompt | llm.with_structured_output(MyModel)   →  MyModel instance
prompt | llm | PydanticOutputParser(...)       →  MyModel instance (prompt-driven)
```

In [ ]:
class ApiEndpoint(BaseModel):
    method: str = Field(description="HTTP method")
    path: str = Field(description="URL path")
    description: str = Field(description="One sentence purpose")


structured_llm = llm.with_structured_output(ApiEndpoint)

endpoint_prompt = ChatPromptTemplate.from_template(
    "Describe one REST endpoint for a notes CRUD API: {action}"
)

structured_chain = endpoint_prompt | structured_llm

endpoint = structured_chain.invoke({"action": "list all notes"})
print(type(endpoint))
print(endpoint)

### 6.1 Choosing Parser vs with_structured_output

| Approach | Pros | Cons |
|----------|------|------|
| **`with_structured_output`** | Strong schema adherence; fewer parse errors | Provider/model support required |
| **`PydanticOutputParser`** | Works on any model; visible format instructions | Model may ignore schema; markdown fences |
| **`JsonOutputParser`** | Flexible dict output | Manual validation optional |
| **`StrOutputParser`** | Simple prose | No structure |

**Recommendation:** prefer **`with_structured_output`** when your model supports it; fall back to **`PydanticOutputParser`** + format instructions for portability.

---

## 7. List Parsers — Comma-Separated and Markdown Lists

When you need a **list of strings** instead of a nested object:

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return only a comma-separated list.\n{format_instructions}"),
    ("human", "List five HTTP status codes for errors."),
]).partial(format_instructions=list_parser.get_format_instructions())

list_chain = list_prompt | llm | list_parser
codes = list_chain.invoke({})
print(codes)
print("type:", type(codes), "len:", len(codes))

In [ ]:
from langchain_core.output_parsers import MarkdownListOutputParser

md_parser = MarkdownListOutputParser()

md_prompt = ChatPromptTemplate.from_template(
    "Give three pytest best practices as a markdown bullet list."
)

md_chain = md_prompt | llm | md_parser
bullets = md_chain.invoke({})
print(bullets)

---

## 8. XMLOutputParser — Structured Tags

**`XMLOutputParser`** expects XML-like tags and returns a nested **dict**. Useful when you want hierarchical structure without JSON.

In [ ]:
from langchain_core.output_parsers import XMLOutputParser

xml_parser = XMLOutputParser()

xml_prompt = ChatPromptTemplate.from_messages([
    ("system", "Respond using XML tags as specified.\n{format_instructions}"),
    ("human", "Extract person: Alice, age 30, role engineer."),
]).partial(format_instructions=xml_parser.get_format_instructions())

xml_chain = xml_prompt | llm | xml_parser
xml_result = xml_chain.invoke({})
print(json.dumps(xml_result, indent=2))

---

## 9. Handling Parse Failures

When the model returns malformed output, parsers raise **`OutputParserException`**. Production chains should catch, log, and optionally retry.

In [ ]:
from langchain_core.exceptions import OutputParserException

bad_json = "Sure! The answer is: not valid json at all"

try:
    json_parser.parse(bad_json)
except OutputParserException as e:
    print("OutputParserException caught:")
    print(str(e)[:200], "...")

### 9.1 Retry Pattern with a Stricter Follow-Up

On failure, call the model again with the error message — a lightweight alternative to **`OutputFixingParser`**:

In [ ]:
def extract_with_retry(text: str, max_attempts: int = 2) -> NoteMeta:
    chain = pydantic_prompt | llm | pydantic_parser
    last_error = ""
    for attempt in range(max_attempts):
        try:
            return chain.invoke({"text": text})
        except OutputParserException as e:
            last_error = str(e)
            text = f"{text}\n\nPrevious output failed validation: {last_error}\nReturn valid JSON only."
    raise OutputParserException(f"Failed after {max_attempts} attempts: {last_error}")


meta = extract_with_retry("Add caching layer. Priority medium. Tags: perf.")
print(meta)

Advanced retry and fallback runnables are covered in **16. Fallbacks and Configurable Runnables**.

---

## 10. Streaming Through Parsers

When you **`stream`** a chain ending in **`StrOutputParser`**, you receive **incremental text chunks** — the parser accumulates and emits parseable partials. Full streaming behavior is explored in **07. Streaming, Batch, and Async**.

In [ ]:
stream_chain = (
    ChatPromptTemplate.from_template("List three facts about {topic} as short bullets.")
    | llm
    | StrOutputParser()
)

print("Streaming parsed text:")
for chunk in stream_chain.stream({"topic": "JWT"}):
    print(chunk, end="", flush=True)
print("\n--- done ---")

**Note:** JSON/Pydantic parsers typically need the **full** completion before parsing — streaming structured output usually requires **`with_structured_output`** or provider streaming schema APIs.

---

## 11. Parsers in RAG Chains

RAG pipelines (**11. RAG with LCEL**) almost always end with **`StrOutputParser`** for user-facing answers. Structured extraction RAG (e.g. return citations + answer object) may use **`with_structured_output`**:

In [ ]:
class RagAnswer(BaseModel):
    answer: str = Field(description="Answer grounded in context")
    confidence: float = Field(description="0.0-1.0 confidence score")
    source_ids: list[str] = Field(description="Chunk ids used, e.g. c1, c3")


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context. Include source_ids from bracket ids.\n{context}"),
    ("human", "{question}"),
])

rag_structured_chain = rag_prompt | llm.with_structured_output(RagAnswer)

rag_result = rag_structured_chain.invoke({
    "context": "[c3] JWT bearer tokens use Authorization: Bearer header.",
    "question": "What header carries the JWT?",
})
print(rag_result)

---

## 12. Testing Parsers Without API Calls

Use **`FakeListChatModel`** (**03. Chat Models and Messages**) to return canned strings and test parser + prompt wiring offline:

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

fake_json_response = json.dumps({
    "title": "Test note",
    "tags": ["test"],
    "priority": 2,
})

fake_llm = FakeListChatModel(responses=[fake_json_response])
test_chain = pydantic_prompt | fake_llm | pydantic_parser

test_note = test_chain.invoke({"text": "ignored — fake model returns fixed JSON"})
print(test_note)

---

## 13. Type Flow — End-to-End

```
dict
  └─► ChatPromptTemplate ──► ChatPromptValue / messages
        └─► ChatOpenAI ──► AIMessage
              └─► StrOutputParser ──► str
              └─► JsonOutputParser ──► dict
              └─► PydanticOutputParser ──► BaseModel
        └─► ChatOpenAI.with_structured_output ──► BaseModel (skips separate parser)
```

Design your API layer to accept the **parser output type**, not `AIMessage`.

---

## 14. Structured Output in Agents

**13. Agents with create_agent`** accepts **`response_format=MyModel`** for typed agent final outputs. That uses structured output under the hood — same Pydantic models as this notebook.

In [ ]:
class ClassifyResult(BaseModel):
    label: str = Field(description="One of: billing, technical, other")
    reason: str = Field(description="Brief justification")


classify_prompt = ChatPromptTemplate.from_template(
    "Classify this support ticket: {ticket}"
)

classify_chain = classify_prompt | llm.with_structured_output(ClassifyResult)
ticket_result = classify_chain.invoke({"ticket": "My JWT token expired after one hour"})
print(ticket_result.label, "—", ticket_result.reason)

---

## 15. Production Guidelines

| Practice | Why |
|----------|-----|
| **Always terminate chains with a parser** | Predictable return types |
| **Prefer `with_structured_output` for schemas** | Fewer parse failures |
| **Embed `format_instructions` via `.partial()`** | Stable prompt, fewer variables |
| **Unit-test parsers with `.parse()`** | No API cost |
| **Log raw model text on parse failure** | Debug schema drift |
| **Validate enums in Pydantic** | Reject invalid labels early |
| **Keep schemas small** | Large schemas increase errors and cost |

In [ ]:
from enum import Enum


class Priority(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"


class TaskTicket(BaseModel):
    title: str
    priority: Priority


task_chain = (
    ChatPromptTemplate.from_template("Extract task from: {text}")
    | llm.with_structured_output(TaskTicket)
)

task = task_chain.invoke({"text": "URGENT: fix production JWT bug"})
print(task.priority, task.title)

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No parser after LLM | `AIMessage` in API response | Add `StrOutputParser` or structured parser |
| JSON parser without format instructions | Invalid JSON from model | Add `get_format_instructions()` to prompt |
| Huge Pydantic schema | Frequent validation errors | Split into smaller extractions |
| Parsing before stream completes | Partial JSON errors | Collect full output first |
| Same schema in prompt and parser drift | Mismatch errors | Generate instructions from one parser instance |
| Ignoring `OutputParserException` | 500 errors in production | Catch, retry, or fallback |
| Using prose parser for machine output | Fragile regex downstream | Use structured output |

---

## 17. Parser Selection Cheat Sheet

| Goal | Parser / API |
|------|----------------|
| Plain text answer | `StrOutputParser()` |
| JSON dict | `JsonOutputParser()` |
| Pydantic model (prompt-driven) | `PydanticOutputParser(pydantic_object=...)` |
| Pydantic model (API-driven) | `llm.with_structured_output(Model)` |
| List of strings | `CommaSeparatedListOutputParser()` |
| Markdown bullets | `MarkdownListOutputParser()` |
| XML-like hierarchy | `XMLOutputParser()` |

---

## 18. Summary

**Output parsers** convert **`AIMessage`** output into **`str`**, **`dict`**, or **Pydantic models** at the chain boundary. **`StrOutputParser`** is the default for text; **`JsonOutputParser`** and **`PydanticOutputParser`** use **`get_format_instructions()`** in prompts; **`with_structured_output`** delegates schema enforcement to the provider API.

Key takeaways:

- Always end chains with a parser for predictable types.
- Pair JSON/Pydantic parsers with **`.partial(format_instructions=...)`** on prompts (**04**).
- Prefer **`with_structured_output`** when supported; fall back to prompt + parser for portability.
- Catch **`OutputParserException`** and retry or log raw completions.
- Test parser logic offline with **`.parse()`** and **`FakeListChatModel`**.

Demonstrations covered string, JSON, Pydantic, list, and XML parsers; structured output chains; parse error handling; streaming text; RAG structured answers; and enum validation.

Next: **06. LCEL Composition Patterns** — `RunnablePassthrough`, `RunnableParallel`, `RunnableBranch`, and advanced dict routing.